# S49_01 — When to Fine-tune

Fine-tuning is frequently the wrong choice. Understanding when it helps (and when it doesn't) saves time and money.

## The hierarchy of options

Try these in order — each is cheaper and faster than the next:

1. **Prompt engineering** — zero-shot, few-shot, CoT
2. **RAG** — add relevant knowledge to the context
3. **Fine-tuning** — adapt model behavior or style
4. **Training from scratch** — almost never the right answer

## What fine-tuning can and cannot do

In [ ]:
# Fine-tuning capability matrix
capabilities = {
    'CAN DO': [
        'Learn a specific output format or style (JSON schemas, markdown templates)',
        'Adopt a brand voice or persona consistently',
        'Improve performance on a narrow, well-defined task',
        'Reduce verbosity / make responses shorter',
        'Learn domain-specific terminology and conventions',
        'Reduce need for long system prompts',
    ],
    'CANNOT DO': [
        'Add new factual knowledge (RAG is better for this)',
        'Reliably update knowledge cutoff',
        'Fix reasoning bugs (a reasoning model beats fine-tuning for math)',
        'Eliminate hallucination without grounding',
        'Teach the model facts it never saw in pre-training',
    ],
}

for category, items in capabilities.items():
    print(f'\n{category}:')
    for item in items:
        print(f'  {'✓' if category == 'CAN DO' else '✗'} {item}')

## Decision framework

In [ ]:
import anthropic

client = anthropic.Anthropic()

# Demonstrate the approach hierarchy with a classification task
review = 'The product broke after 2 weeks. Terrible quality.'

# Option 1: Zero-shot prompt
def classify_zero_shot(text):
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=16,
        messages=[{'role': 'user', 'content': f'Classify as POSITIVE or NEGATIVE: "{text}"'}],
    )
    return msg.content[0].text.strip()

# Option 2: Strong system prompt
def classify_with_system(text):
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=16,
        system='You are a sentiment classifier. Respond with exactly one word: POSITIVE, NEGATIVE, or NEUTRAL.',
        messages=[{'role': 'user', 'content': text}],
    )
    return msg.content[0].text.strip()

print('Zero-shot:', classify_zero_shot(review))
print('System prompt:', classify_with_system(review))
print('\nIf these work → no fine-tuning needed')
print('Fine-tune only if: (1) accuracy is still insufficient, or (2) you need low latency/cost at massive scale')

## Data requirements

In [ ]:
# Minimum data thresholds (rough rules of thumb)
data_requirements = [
    ('Style/format adaptation', '50–200 examples', 'Low'),
    ('Classification task', '500–2000 examples', 'Medium'),
    ('Instruction following (SFT)', '1k–10k examples', 'Medium'),
    ('Domain specialization', '5k–50k examples', 'Medium-High'),
    ('RLHF/DPO alignment', '1k–100k preference pairs', 'High'),
    ('Pre-training from scratch', 'Billions of tokens', 'Extreme'),
]

print(f'{'Task':35} {'Data needed':25} {'Cost'}') 
print('-' * 70)
for task, data, cost in data_requirements:
    print(f'{task:35} {data:25} {cost}')

## Fine-tuning approaches

| Method | Params updated | VRAM needed | Quality |
|--------|---------------|-------------|--------|
| Full fine-tune | All ~7B+ | 80GB+ | Best |
| LoRA | ~0.3% | 16–24GB | Near-best |
| QLoRA (4-bit) | ~0.3% | 8–16GB | Good |
| API fine-tuning (OpenAI/Anthropic) | Managed | None | Good, opaque |

For most practitioners: **QLoRA via Unsloth on a single GPU** is the sweet spot.

Next: [S49_02_lora_qlora.ipynb](./S49_02_lora_qlora.ipynb)